# Clustering — Zonas calientes (no supervisado, dos niveles)

Pieza central del entregable de **machine learning no supervisado**. Descubrimos
**zonas calientes** (alta demanda y/o baja oferta) *sin usar etiquetas* y luego
validamos contra el *ground-truth* sintético sólo para medir la calidad.

**Dos niveles de datos:**
- **Nivel A** (`solicitudes_profesor.csv`): cada solicitud cruda `(lat, lng, ...)`.
  Sirve de **baseline geométrico** (DBSCAN haversine sobre puntos).
- **Nivel B** (`demanda_agregada_profesor.csv`): una fila por **celda × franja ×
  municipio**, con `demand_density` y `supply_demand_ratio`. Es la **entrada del
  modelo**: agrupa celdas vecinas de alta demanda en zonas operativas.

**Sólo no supervisado:** `DBSCAN`, `NearestNeighbors`, `LocalOutlierFactor`,
`KMeans`. Nunca `KNeighborsClassifier`. El *ground-truth* (`zone_id_true`,
`is_hot_true`) **jamás** entra como feature: sólo se usa al final para ARI / NMI /
precisión-recall.

**Por municipio:** todo el pipeline se ajusta **por municipio** (cada uno tiene su
propia geografía y escala de demanda).

**Secciones**
1. Configuración, carga y selección del bucket pico.
2. Baseline — DBSCAN haversine sobre puntos crudos (Nivel A).
3. Modelo — matriz de features de las celdas (Nivel B) + escalado.
4. Calibración de `eps` con kNN no supervisado (gráfico de k-distancias).
5. DBSCAN sobre Nivel B (zonas calientes).
6. LOF no supervisado vs ruido de DBSCAN.
7. KMeans baseline (k por codo + silhouette).
8. Validación vs *ground-truth* (ARI / NMI / precisión-recall / silhouette).
9. Extracción de zonas calientes (tabla + scatter final).

Notas de diseño: `documentation/04-notas-clustering.md`.

## 1. Configuración, carga y selección del bucket pico

Trabajamos sobre un `(dia_tipo, hour)` **representativo de hora pico**. Elegimos
`entre_semana`, hora **18** (salida laboral): concentra mucha demanda y celdas
calientes en ambos municipios. El clustering se corre **por municipio** sobre las
celdas de ese bucket.

In [1]:
import matplotlib
matplotlib.use("Agg")   # backend no interactivo para nbconvert (headless)

from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    precision_score,
    recall_score,
    f1_score,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110

# Rutas (idénticas al resto de notebooks)
NOTEBOOK_DIR = Path(".").resolve()
ROOT_DIR = NOTEBOOK_DIR.parent
DATA_DIR = ROOT_DIR / "documentation"
FIG_DIR = ROOT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

R_TIERRA_M = 6_371_000.0          # radio terrestre (m) → eps haversine en radianes
MUNICIPIOS = [1, 2]
DIA_PICO = "entre_semana"
HORA_PICO = 18
SEED = 42

# Bounding boxes por municipio (del config de generación)
BBOX = {
    1: {"lat": (16.70, 16.80), "lng": (-93.16, -93.08)},
    2: {"lat": (16.86, 16.94), "lng": (-92.12, -92.04)},
}


def _save(fig: plt.Figure, nombre: str) -> None:
    """Guarda `fig` en figures/<nombre> y la cierra."""
    ruta = FIG_DIR / nombre
    fig.savefig(ruta, bbox_inches="tight", dpi=120)
    print(f"Guardada: {ruta.name}")
    plt.close(fig)


# Nivel A — solicitudes crudas (baseline de puntos)
df_a = pd.read_csv(DATA_DIR / "solicitudes_profesor.csv")
df_a["accepted"] = df_a["accepted"].astype("boolean")

# Nivel B — demanda agregada (entrada del modelo)
df_b = pd.read_csv(DATA_DIR / "demanda_agregada_profesor.csv")
df_b["is_hot_true"] = df_b["is_hot_true"].astype(bool)

print(f"Nivel A — {len(df_a):,} solicitudes, {df_a.shape[1]} columnas")
print(f"Nivel B — {len(df_b):,} celdas, {df_b.shape[1]} columnas")
print(f"Bucket pico: dia_tipo={DIA_PICO!r}, hour={HORA_PICO}")

Nivel A — 48,783 solicitudes, 14 columnas
Nivel B — 5,357 celdas, 14 columnas
Bucket pico: dia_tipo='entre_semana', hour=18


In [2]:
# Resumen del bucket pico por municipio (Nivel B)
resumen = []
for muni in MUNICIPIOS:
    sub = df_b[
        (df_b["municipio"] == muni)
        & (df_b["dia_tipo"] == DIA_PICO)
        & (df_b["hour"] == HORA_PICO)
    ]
    resumen.append({
        "municipio": muni,
        "celdas": len(sub),
        "celdas_hot_true": int(sub["is_hot_true"].sum()),
        "zonas_true": sorted(z for z in sub["zone_id_true"].unique() if z >= 0),
        "demand_density_max": round(float(sub["demand_density"].max()), 1),
        "ratio_min": round(float(sub["supply_demand_ratio"].min()), 2),
    })
pd.DataFrame(resumen)

,municipio,celdas,celdas_hot_true,zonas_true,demand_density_max,ratio_min
0,1,134,19,"[0, 1, 2]",3166.7,0.0
1,2,138,13,"[0, 1]",4544.4,0.0


## 2. Baseline — DBSCAN haversine sobre puntos crudos (Nivel A)

**Comparación geométrica.** Sobre las solicitudes del bucket pico aplicamos
`DBSCAN(metric="haversine")` con las coordenadas en **radianes** y **sin escalar**:
es densidad espacial pura. `eps` se fija en metros y se convierte a radianes
(`eps_rad = metros / 6_371_000`). Esto recupera dónde se concentran *físicamente*
las solicitudes, pero **sólo usa ubicación**: no sabe nada de demanda ni de
oferta/demanda.

In [3]:
def solicitudes_bucket(muni: int) -> pd.DataFrame:
    """Solicitudes válidas (dentro del bbox) del bucket pico para un municipio."""
    bb = BBOX[muni]
    s = df_a[
        (df_a["municipio"] == muni)
        & (df_a["day_of_week"] < 5)        # entre semana
        & (df_a["hour"] == HORA_PICO)
        & df_a["lat"].between(*bb["lat"])  # descarta anomalías fuera de bbox
        & df_a["lng"].between(*bb["lng"])
    ].dropna(subset=["lat", "lng"]).copy()
    return s


def dbscan_haversine(s: pd.DataFrame, eps_m: float, min_samples: int = 8) -> np.ndarray:
    """DBSCAN espacial puro: coords en radianes, eps en radianes = metros / R."""
    coords_rad = np.radians(s[["lat", "lng"]].to_numpy())
    eps_rad = eps_m / R_TIERRA_M
    return DBSCAN(eps=eps_rad, min_samples=min_samples, metric="haversine").fit_predict(coords_rad)


EPS_BASE_M = 250.0   # ~ tamaño de celda; agrupa solicitudes dentro de una zona

baseline_a: dict[int, dict] = {}
for muni in MUNICIPIOS:
    s = solicitudes_bucket(muni)
    lab = dbscan_haversine(s, EPS_BASE_M)
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    n_noise = int((lab == -1).sum())
    # ARI/NMI contra zona real (sólo solicitudes con zona >= 0)
    m = s["zone_id_true"].to_numpy() >= 0
    ari = adjusted_rand_score(s.loc[m, "zone_id_true"], lab[m])
    nmi = normalized_mutual_info_score(s.loc[m, "zone_id_true"], lab[m])
    baseline_a[muni] = {"s": s, "lab": lab, "n_clusters": n_clusters,
                        "n_noise": n_noise, "ari": ari, "nmi": nmi}
    print(f"muni {muni}: {len(s):,} solicitudes → {n_clusters} clusters, "
          f"{n_noise} ruido ({n_noise/len(s)*100:.0f}%) | ARI={ari:.3f} NMI={nmi:.3f}")

muni 1: 1,611 solicitudes → 3 clusters, 109 ruido (7%) | ARI=1.000 NMI=1.000
muni 2: 1,616 solicitudes → 2 clusters, 129 ruido (8%) | ARI=1.000 NMI=1.000


In [4]:
# Scatter del baseline de puntos por municipio
fig, axes = plt.subplots(1, len(MUNICIPIOS), figsize=(13, 5.5))
for ax, muni in zip(np.atleast_1d(axes), MUNICIPIOS):
    info = baseline_a[muni]
    s, lab = info["s"], info["lab"]
    ruido = lab == -1
    ax.scatter(s.loc[ruido, "lng"], s.loc[ruido, "lat"], s=8, c="lightgray",
               label="ruido", alpha=0.6)
    ax.scatter(s.loc[~ruido, "lng"], s.loc[~ruido, "lat"], s=10,
               c=lab[~ruido], cmap="tab10", alpha=0.7)
    ax.set_title(f"Muni {muni} — baseline puntos (haversine {EPS_BASE_M:.0f} m)\n"
                 f"{info['n_clusters']} clusters, ARI={info['ari']:.2f}")
    ax.set_xlabel("lng"); ax.set_ylabel("lat")
    ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
_save(fig, "clu_baseline_puntos.png")

Guardada: clu_baseline_puntos.png


**Lectura.** El baseline recupera las zonas plantadas casi a la perfección
(ARI ≈ 1.0): es un problema *geométrico* fácil porque las solicitudes son gaussianas
alrededor de centros conocidos. Pero responde la pregunta equivocada para el negocio:
nos dice **dónde caen puntos**, no **dónde la demanda supera a la oferta**. Para eso
necesitamos el Nivel B.

## 3. Modelo — matriz de features de las celdas (Nivel B)

Para cada celda del bucket pico construimos el vector

$$x = [\,\text{centroid\_lat},\ \text{centroid\_lon},\ \text{demand\_density},\ \text{supply\_demand\_ratio}\,]$$

Mezcla **ubicación** (para que el cluster sea espacialmente contiguo) con **demanda**
(`demand_density`) y **presión oferta/demanda** (`supply_demand_ratio`, bajo = demanda
> oferta). Aquí **sí escalamos con `StandardScaler` las cuatro features**, a diferencia
del baseline haversine: al combinar grados de latitud/longitud (rango ~0.1) con
densidades (rango ~10–4500) en una **misma distancia euclidiana**, sin estandarizar la
demanda dominaría por completo y la geografía sería irrelevante. La estandarización
pone las cuatro en la misma escala (media 0, desviación 1). *(En el baseline NO se
escala porque allí la distancia es haversine pura sobre coordenadas — no se mezcla con
otras unidades.)*

In [5]:
FEATURES = ["centroid_lat", "centroid_lon", "demand_density", "supply_demand_ratio"]


def celdas_bucket(muni: int) -> pd.DataFrame:
    """Celdas del bucket pico de un municipio (Nivel B)."""
    return df_b[
        (df_b["municipio"] == muni)
        & (df_b["dia_tipo"] == DIA_PICO)
        & (df_b["hour"] == HORA_PICO)
    ].reset_index(drop=True).copy()


def matriz_escalada(celdas: pd.DataFrame) -> np.ndarray:
    """Matriz de features estandarizada (StandardScaler a las 4 columnas)."""
    X = celdas[FEATURES].to_numpy(dtype=float)
    return StandardScaler().fit_transform(X)


# Sanity check: rangos antes / después de escalar (muni 1)
celdas_demo = celdas_bucket(1)
X_demo = celdas_demo[FEATURES].to_numpy(float)
Xs_demo = matriz_escalada(celdas_demo)
comp = pd.DataFrame({
    "feature": FEATURES,
    "min_orig": X_demo.min(0).round(2), "max_orig": X_demo.max(0).round(2),
    "min_esc": Xs_demo.min(0).round(2), "max_esc": Xs_demo.max(0).round(2),
})
print("Muni 1 — escalado de features:")
comp

Muni 1 — escalado de features:


,feature,min_orig,max_orig,min_esc,max_esc
0,centroid_lat,16.70,16.80,-1.73,1.90
1,centroid_lon,-93.16,-93.08,-1.76,1.93
2,demand_density,11.11,3166.67,-0.28,6.88
3,supply_demand_ratio,0.00,8.00,-1.48,2.33


## 4. Calibración de `eps` con kNN no supervisado (k-distancias)

DBSCAN necesita `eps`. Lo calibramos **sin etiquetas** con el método clásico de
*k-distancias*: para `min_samples` vecinos calculamos, para cada celda, la distancia
a su k-ésimo vecino más cercano (`NearestNeighbors`), las ordenamos y buscamos el
**codo** de la curva: el `eps` está donde la curva se dispara (las distancias pasan de
"dentro de un grupo denso" a "entre grupos").

Detectamos el codo con la **máxima distancia por debajo de la cuerda** que une los
extremos de la curva ordenada. Recortamos el **2 % superior** de la cola antes de
medir: unas pocas celdas extremadamente aisladas (ej. un único ratio atípico) tuercen
la cuerda y empujarían `eps` a un valor absurdo. Este recorte hace la calibración
estable en ambos municipios.

In [6]:
MIN_SAMPLES = 4   # densidad mínima de un cluster en el espacio de 4 features


def curva_k_distancias(Xs: np.ndarray, min_samples: int) -> np.ndarray:
    """Distancia ordenada al (min_samples)-ésimo vecino más cercano de cada punto."""
    nn = NearestNeighbors(n_neighbors=min_samples).fit(Xs)
    dist, _ = nn.kneighbors(Xs)
    return np.sort(dist[:, -1])


def codo_k_distancias(kdist: np.ndarray, recorte: float = 0.02) -> tuple[int, float]:
    """Codo = máx distancia por debajo de la cuerda, recortando la cola superior.

    Devuelve (índice del codo, eps). Robusto ante celdas extremadamente aisladas.
    """
    n = len(kdist)
    hi = max(3, int(np.ceil(n * (1 - recorte))))   # ignora el 'recorte' superior
    k = kdist[:hi]
    x = np.arange(len(k), dtype=float)
    x1, y1, x2, y2 = 0.0, k[0], len(k) - 1.0, k[-1]
    # distancia firmada de cada punto a la recta extremo-a-extremo (cuerda)
    d = ((y2 - y1) * x - (x2 - x1) * k + x2 * y1 - y2 * x1) / np.hypot(y2 - y1, x2 - x1)
    idx = int(np.argmin(d))   # más negativo = más por debajo de la cuerda = codo
    return idx, float(kdist[idx])


eps_por_muni: dict[int, float] = {}
fig, axes = plt.subplots(1, len(MUNICIPIOS), figsize=(13, 5))
for ax, muni in zip(np.atleast_1d(axes), MUNICIPIOS):
    Xs = matriz_escalada(celdas_bucket(muni))
    kdist = curva_k_distancias(Xs, MIN_SAMPLES)
    idx, eps = codo_k_distancias(kdist)
    eps_por_muni[muni] = eps
    ax.plot(kdist, lw=1.8, color="steelblue")
    ax.axhline(eps, ls="--", color="crimson")
    ax.scatter([idx], [eps], color="crimson", zorder=5)
    ax.set_title(f"Muni {muni} — k-distancias (k={MIN_SAMPLES})\ncodo → eps={eps:.3f}")
    ax.set_xlabel("celdas ordenadas")
    ax.set_ylabel(f"dist. al {MIN_SAMPLES}º vecino (escalada)")
    ax.annotate(f"eps={eps:.3f}", (idx, eps), textcoords="offset points",
                xytext=(10, 10), color="crimson")
plt.tight_layout()
_save(fig, "clu_k_distancias.png")
print("eps calibrado por municipio:", {m: round(e, 3) for m, e in eps_por_muni.items()})

Guardada: clu_k_distancias.png
eps calibrado por municipio: {1: 0.42, 2: 0.394}


## 5. DBSCAN sobre Nivel B — zonas calientes

Con el `eps` calibrado corremos `DBSCAN(metric="euclidean")` sobre la matriz escalada.
Las celdas que quedan en un cluster (`label != -1`) son **candidatas a zona caliente**:
celdas vecinas con demanda/oferta parecidas. El ruido (`-1`) son celdas que no forman
densidad — la mayoría son el "fondo" de celdas de una sola solicitud.

In [7]:
def correr_dbscan_b(muni: int) -> dict:
    """DBSCAN Nivel B para un municipio. Devuelve celdas + etiquetas + métricas base."""
    celdas = celdas_bucket(muni)
    Xs = matriz_escalada(celdas)
    eps = eps_por_muni[muni]
    lab = DBSCAN(eps=eps, min_samples=MIN_SAMPLES, metric="euclidean").fit_predict(Xs)
    celdas = celdas.assign(cluster=lab)
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    n_noise = int((lab == -1).sum())
    return {"celdas": celdas, "Xs": Xs, "lab": lab, "eps": eps,
            "n_clusters": n_clusters, "n_noise": n_noise}


dbscan_b: dict[int, dict] = {}
for muni in MUNICIPIOS:
    r = correr_dbscan_b(muni)
    dbscan_b[muni] = r
    print(f"muni {muni}: eps={r['eps']:.3f} → {r['n_clusters']} clusters, "
          f"{r['n_noise']} ruido de {len(r['celdas'])} celdas")

muni 1: eps=0.420 → 2 clusters, 123 ruido de 134 celdas
muni 2: eps=0.394 → 5 clusters, 117 ruido de 138 celdas


In [8]:
# Visualización: celdas coloreadas por cluster DBSCAN (Nivel B)
fig, axes = plt.subplots(1, len(MUNICIPIOS), figsize=(13, 5.5))
for ax, muni in zip(np.atleast_1d(axes), MUNICIPIOS):
    celdas = dbscan_b[muni]["celdas"]
    lab = dbscan_b[muni]["lab"]
    ruido = lab == -1
    ax.scatter(celdas.loc[ruido, "centroid_lon"], celdas.loc[ruido, "centroid_lat"],
               s=30, c="lightgray", marker="s", alpha=0.5, label="ruido (-1)")
    sc = ax.scatter(celdas.loc[~ruido, "centroid_lon"], celdas.loc[~ruido, "centroid_lat"],
                    s=60, c=lab[~ruido], cmap="tab10", marker="s",
                    edgecolor="k", linewidth=0.3)
    ax.set_title(f"Muni {muni} — DBSCAN Nivel B\n"
                 f"{dbscan_b[muni]['n_clusters']} clusters (eps={dbscan_b[muni]['eps']:.2f})")
    ax.set_xlabel("centroid_lon"); ax.set_ylabel("centroid_lat")
    ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
_save(fig, "clu_dbscan_nivel_b.png")

Guardada: clu_dbscan_nivel_b.png

## 6. LOF no supervisado vs ruido de DBSCAN

`LocalOutlierFactor` puntúa cada celda por cuánto se desvía su densidad local de la de
sus vecinas. En un único bucket, las celdas verdaderamente calientes son **pocas y de
valores extremos** (densidad muy alta, ratio muy bajo): se comportan como **anomalías**
más que como un cluster denso. Por eso LOF es un detector natural de zonas calientes,
y esperamos que sus *outliers* coincidan en buena medida con el **ruido** de DBSCAN.

In [9]:
lof_b: dict[int, dict] = {}
for muni in MUNICIPIOS:
    Xs = dbscan_b[muni]["Xs"]
    lab_db = dbscan_b[muni]["lab"]
    n = len(Xs)
    lof = LocalOutlierFactor(n_neighbors=min(20, n - 1))
    lab_lof = lof.fit_predict(Xs)            # -1 = outlier
    es_out = lab_lof == -1
    es_ruido_db = lab_db == -1
    overlap = int((es_out & es_ruido_db).sum())
    lof_b[muni] = {"lab_lof": lab_lof, "n_out": int(es_out.sum()), "overlap": overlap}
    print(f"muni {muni}: LOF outliers={int(es_out.sum())}, "
          f"ruido DBSCAN={int(es_ruido_db.sum())}, "
          f"coinciden={overlap}/{int(es_out.sum())} de los outliers")

muni 1: LOF outliers=7, ruido DBSCAN=123, coinciden=7/7 de los outliers
muni 2: LOF outliers=5, ruido DBSCAN=117, coinciden=5/5 de los outliers


In [10]:
# Comparación visual: outliers LOF vs ruido DBSCAN (muni con más celdas hot)
fig, axes = plt.subplots(1, len(MUNICIPIOS), figsize=(13, 5.5))
for ax, muni in zip(np.atleast_1d(axes), MUNICIPIOS):
    celdas = dbscan_b[muni]["celdas"]
    es_out = lof_b[muni]["lab_lof"] == -1
    ax.scatter(celdas["centroid_lon"], celdas["centroid_lat"], s=30,
               c="lightgray", marker="s", alpha=0.5, label="normal")
    ax.scatter(celdas.loc[es_out, "centroid_lon"], celdas.loc[es_out, "centroid_lat"],
               s=80, facecolor="none", edgecolor="crimson", linewidth=1.5,
               marker="s", label="outlier LOF")
    # celdas hot reales (para contraste visual)
    hot = celdas["is_hot_true"].to_numpy()
    ax.scatter(celdas.loc[hot, "centroid_lon"], celdas.loc[hot, "centroid_lat"],
               s=12, c="black", marker="x", label="hot real")
    ax.set_title(f"Muni {muni} — LOF outliers vs hot real")
    ax.set_xlabel("centroid_lon"); ax.set_ylabel("centroid_lat")
    ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
_save(fig, "clu_lof_vs_dbscan.png")

Guardada: clu_lof_vs_dbscan.png


## 7. KMeans baseline (k por codo + silhouette)

`KMeans` particiona **todas** las celdas en `k` grupos (no tiene noción de ruido).
Elegimos `k` combinando el **codo de la inercia** y el **silhouette** (ambos no
supervisados). Es un baseline para contrastar con DBSCAN: KMeans siempre asigna todas
las celdas, así que mezcla el fondo de baja demanda con las celdas calientes.

In [11]:
kmeans_b: dict[int, dict] = {}
fig, axes = plt.subplots(1, len(MUNICIPIOS), figsize=(13, 5))
for ax, muni in zip(np.atleast_1d(axes), MUNICIPIOS):
    Xs = dbscan_b[muni]["Xs"]
    ks = list(range(2, 9))
    inercias, sils = [], []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xs)
        inercias.append(km.inertia_)
        sils.append(silhouette_score(Xs, km.labels_))
    best_k = ks[int(np.argmax(sils))]
    km = KMeans(n_clusters=best_k, n_init=10, random_state=SEED).fit(Xs)
    kmeans_b[muni] = {"best_k": best_k, "sil": max(sils), "lab": km.labels_}
    ax.plot(ks, inercias, "o-", color="steelblue", label="inercia (codo)")
    ax.set_xlabel("k"); ax.set_ylabel("inercia", color="steelblue")
    ax2 = ax.twinx()
    ax2.plot(ks, sils, "s--", color="darkorange", label="silhouette")
    ax2.set_ylabel("silhouette", color="darkorange")
    ax.axvline(best_k, ls=":", color="crimson")
    ax.set_title(f"Muni {muni} — KMeans: k*={best_k} (sil={max(sils):.2f})")
plt.tight_layout()
_save(fig, "clu_kmeans_seleccion_k.png")
print("KMeans k* por municipio:", {m: kmeans_b[m]["best_k"] for m in MUNICIPIOS})

Guardada: clu_kmeans_seleccion_k.png
KMeans k* por municipio: {1: 6, 2: 2}


## 8. Validación vs *ground-truth* (ARI / NMI / precisión-recall)

Ahora —y sólo ahora— usamos las etiquetas ocultas para **medir**, no para entrenar:

- **ARI** y **NMI** entre los clusters de DBSCAN y `zone_id_true`, sobre las celdas con
  zona real ≥ 0 (mide si los clusters recuperan las zonas plantadas).
- **Precisión / recall / F1** tratando `cluster != -1` como predicción de "celda
  caliente" frente a `is_hot_true`.
- **Silhouette** de la solución DBSCAN (cohesión interna, no supervisado).

Comparamos contra el baseline de puntos (Nivel A) y contra KMeans.

In [12]:
def metricas_dbscan(muni: int) -> dict:
    """ARI/NMI/precision/recall/F1/silhouette de la solución DBSCAN-B del municipio."""
    info = dbscan_b[muni]
    celdas, lab, Xs = info["celdas"], info["lab"], info["Xs"]
    yt_hot = celdas["is_hot_true"].to_numpy()
    pred_hot = lab != -1
    m = celdas["zone_id_true"].to_numpy() >= 0     # celdas con zona real
    ari = adjusted_rand_score(celdas.loc[m, "zone_id_true"], lab[m]) if m.sum() > 1 else float("nan")
    nmi = normalized_mutual_info_score(celdas.loc[m, "zone_id_true"], lab[m]) if m.sum() > 1 else float("nan")
    n_clusters = info["n_clusters"]
    sil = (silhouette_score(Xs, lab)
           if n_clusters > 1 and (lab != -1).sum() > n_clusters else float("nan"))
    return {
        "municipio": muni,
        "clusters": n_clusters,
        "ruido": info["n_noise"],
        "ARI_zona": round(ari, 3),
        "NMI_zona": round(nmi, 3),
        "precision_hot": round(precision_score(yt_hot, pred_hot, zero_division=0), 3),
        "recall_hot": round(recall_score(yt_hot, pred_hot, zero_division=0), 3),
        "f1_hot": round(f1_score(yt_hot, pred_hot, zero_division=0), 3),
        "silhouette": round(sil, 3) if sil == sil else float("nan"),
    }


tabla_val = pd.DataFrame([metricas_dbscan(m) for m in MUNICIPIOS])
print("Validación DBSCAN Nivel B por municipio:")
tabla_val

Validación DBSCAN Nivel B por municipio:


,municipio,clusters,ruido,ARI_zona,NMI_zona,precision_hot,recall_hot,f1_hot,silhouette
0,1,2,123,0.193,0.430,0.727,0.421,0.533,-0.113
1,2,5,117,0.149,0.367,0.333,0.538,0.412,-0.200


In [13]:
# Comparación: baseline puntos (A) vs DBSCAN Nivel B, métrica ARI vs zona real
comparacion = []
for muni in MUNICIPIOS:
    comparacion.append({
        "municipio": muni,
        "baseline_A_ARI": round(baseline_a[muni]["ari"], 3),
        "baseline_A_NMI": round(baseline_a[muni]["nmi"], 3),
        "dbscan_B_ARI": tabla_val.loc[tabla_val.municipio == muni, "ARI_zona"].iloc[0],
        "dbscan_B_NMI": tabla_val.loc[tabla_val.municipio == muni, "NMI_zona"].iloc[0],
        "dbscan_B_precision_hot": tabla_val.loc[tabla_val.municipio == muni, "precision_hot"].iloc[0],
        "dbscan_B_recall_hot": tabla_val.loc[tabla_val.municipio == muni, "recall_hot"].iloc[0],
        "kmeans_k": kmeans_b[muni]["best_k"],
        "kmeans_silhouette": round(kmeans_b[muni]["sil"], 3),
    })
df_comp = pd.DataFrame(comparacion)
print("Baseline A (geometría pura) vs DBSCAN B (demanda+oferta) vs KMeans:")
df_comp

Baseline A (geometría pura) vs DBSCAN B (demanda+oferta) vs KMeans:


,municipio,baseline_A_ARI,baseline_A_NMI,dbscan_B_ARI,dbscan_B_NMI,dbscan_B_precision_hot,dbscan_B_recall_hot,kmeans_k,kmeans_silhouette
0,1,1.0,1.0,0.193,0.430,0.727,0.421,6,0.304
1,2,1.0,1.0,0.149,0.367,0.333,0.538,2,0.608


**Interpretación de la validación.**

- El **baseline de puntos (Nivel A)** logra ARI ≈ 1.0 porque recupera la **geometría**
  de zonas gaussianas — pero es un problema de juguete: sólo agrupa *ubicación* y no
  distingue una zona caliente de una fría.
- **DBSCAN Nivel B** trabaja sobre la superficie de demanda agregada (mucho más ruidosa:
  un fondo enorme de celdas de una sola solicitud). Su ARI/NMI es menor en términos
  absolutos, pero **sus clusters tienen significado operativo**: agrupan celdas por
  demanda y presión oferta/demanda, que es justo lo que el conductor necesita. La
  **precisión de celdas hot** confirma que cuando DBSCAN marca un cluster, suele ser
  efectivamente caliente.
- **KMeans** alcanza un silhouette alto separando "fondo vs picos", pero al forzar a
  *todas* las celdas a un cluster no aísla las calientes tan limpiamente como DBSCAN
  (que manda el fondo a ruido).

## 9. Extracción de zonas calientes (tabla + scatter final)

Por cada cluster de DBSCAN resumimos lo que el microservicio expondría al conductor:
**centro** (centroide de las celdas), `n_requests` **total**, `demand_density`
**media**, `supply_demand_ratio` **medio** y fracción de celdas que el *ground-truth*
marca como calientes. Ordenamos por demanda total: arriba quedan las zonas más
calientes.

In [14]:
def extraer_zonas(muni: int) -> pd.DataFrame:
    """Resumen por cluster DBSCAN (excluye ruido) = zonas calientes del municipio."""
    celdas = dbscan_b[muni]["celdas"]
    filas = []
    for cl in sorted(c for c in celdas["cluster"].unique() if c != -1):
        g = celdas[celdas["cluster"] == cl]
        filas.append({
            "municipio": muni,
            "cluster": int(cl),
            "n_celdas": len(g),
            "centro_lat": round(float(g["centroid_lat"].mean()), 5),
            "centro_lon": round(float(g["centroid_lon"].mean()), 5),
            "n_requests_total": int(g["n_requests"].sum()),
            "demand_density_media": round(float(g["demand_density"].mean()), 1),
            "supply_demand_ratio_medio": round(float(g["supply_demand_ratio"].mean()), 2),
            "frac_hot_true": round(float(g["is_hot_true"].mean()), 2),
        })
    cols = ["municipio", "cluster", "n_celdas", "centro_lat", "centro_lon",
            "n_requests_total", "demand_density_media",
            "supply_demand_ratio_medio", "frac_hot_true"]
    if not filas:
        return pd.DataFrame(columns=cols)
    return (pd.DataFrame(filas)
            .sort_values("n_requests_total", ascending=False)
            .reset_index(drop=True))


zonas_calientes = pd.concat([extraer_zonas(m) for m in MUNICIPIOS], ignore_index=True)
print(f"Zonas calientes extraídas (total): {len(zonas_calientes)}")
zonas_calientes

Zonas calientes extraídas (total): 7


,municipio,cluster,n_celdas,centro_lat,centro_lon,n_requests_total,demand_density_media,supply_demand_ratio_medio,frac_hot_true
0,1,0,7,16.73604,-93.11124,114,181.0,0.31,0.71
1,1,1,4,16.75328,-93.11481,40,111.1,0.46,0.75
2,2,3,4,16.90991,-92.07009,176,488.9,0.10,1.00
3,2,1,5,16.89966,-92.08142,53,117.8,0.46,0.60
4,2,0,4,16.88900,-92.09100,4,11.1,2.00,0.00
5,2,2,4,16.90182,-92.10651,4,11.1,4.00,0.00
6,2,4,4,16.91733,-92.11663,4,11.1,6.00,0.00


In [15]:
# Scatter final: celdas (color = demand_density) con las zonas calientes marcadas
fig, axes = plt.subplots(1, len(MUNICIPIOS), figsize=(13, 5.5))
for ax, muni in zip(np.atleast_1d(axes), MUNICIPIOS):
    celdas = dbscan_b[muni]["celdas"]
    sc = ax.scatter(celdas["centroid_lon"], celdas["centroid_lat"],
                    c=celdas["demand_density"], cmap="viridis",
                    s=45, marker="s", norm="log", alpha=0.85)
    zc = zonas_calientes[zonas_calientes["municipio"] == muni]
    ax.scatter(zc["centro_lon"], zc["centro_lat"], s=260, marker="*",
               facecolor="crimson", edgecolor="white", linewidth=0.8,
               label="centro zona caliente", zorder=5)
    ax.set_title(f"Muni {muni} — demanda y zonas calientes")
    ax.set_xlabel("centroid_lon"); ax.set_ylabel("centroid_lat")
    ax.legend(loc="upper right", fontsize=8)
    fig.colorbar(sc, ax=ax, label="demand_density (log)")
plt.tight_layout()
_save(fig, "clu_zonas_calientes_final.png")

Guardada: clu_zonas_calientes_final.png


### Resumen

- **Baseline (Nivel A, haversine):** recupera la geometría de las zonas plantadas
  (ARI ≈ 1.0) pero sólo usa ubicación → no identifica zonas *calientes*, sólo densas.
- **Modelo (Nivel B, DBSCAN sobre `[lat, lon, demand_density, supply_demand_ratio]`):**
  agrupa celdas vecinas por **demanda y presión oferta/demanda**, con `eps` calibrado
  por kNN no supervisado. Sus clusters son las **zonas calientes** que el microservicio
  expone, con precisión-recall razonable frente a `is_hot_true`.
- **LOF** confirma que las celdas calientes se comportan como **anomalías** en un único
  bucket, solapando con el ruido de DBSCAN.
- **KMeans** sirve de baseline particional; no aísla el fondo tan bien como DBSCAN.

Los valores numéricos de ARI/NMI/precisión-recall por municipio (tablas de la sección 8)
quedan documentados en `documentation/04-notas-clustering.md`.